In [ ]:
import time
from collections import Counter

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import folium
import requests
from tqdm.notebook import tqdm
from neo4j import GraphDatabase

from dotenv import load_dotenv
import os

In [ ]:
N_SAMPLE   = 2500
GRID_STEP  = 0.01
K_NEAREST  = 5
SLEEP_SEC  = 0.05
PERCENTILE_LOW  = 25
GEOJSON    = "lombardia.geojson"
OUTPUT_MAP = "RouteHub.html"

OSRM_BASE = "https://router.project-osrm.org"

load_dotenv()

NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_USER     = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)
driver.verify_connectivity()
print(f"Connesso a {NEO4J_URI}")

def run(cypher, **params):
    with driver.session(database=NEO4J_DATABASE) as s:
        return pd.DataFrame([r.data() for r in s.run(cypher, **params)])

In [ ]:
regione = gpd.read_file(GEOJSON)

points = regione.dissolve().sample_points(N_SAMPLE)

sample = [{"lat": p.y, "lon": p.x} for p in points.iloc[0].geoms]

In [ ]:
hospitals_df = run("""
    MATCH (o:Hub)-[:HA_AREA]->(a:AreaSpecialistica) 
    WHERE o.Coordinate IS NOT NULL AND a.Nome = "PRONTO SOCCORSO"
    RETURN
        o.Nome AS Nome,
        o.Coordinate.latitude  AS lat,
        o.Coordinate.longitude AS lon
""")

hosp_lats = hospitals_df["lat"].to_numpy()
hosp_lons = hospitals_df["lon"].to_numpy()

In [ ]:
def osrm_table(origin_lat, origin_lon, candidates: pd.DataFrame):
    coord_str = f"{origin_lon},{origin_lat}"
    for _, r in candidates.iterrows():
        coord_str += f";{r['lon']},{r['lat']}"
    dest_str = ";".join(str(i + 1) for i in range(len(candidates)))
    url = (
        f"{OSRM_BASE}/table/v1/driving/{coord_str}"
        f"?sources=0&destinations={dest_str}&annotations=duration"
    )
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    if data.get("code") != "Ok":
        return [None] * len(candidates)
    return data["durations"][0]

def osrm_route(origin_lat, origin_lon, dest_lat, dest_lon):
    url = (
        f"{OSRM_BASE}/route/v1/driving/"
        f"{origin_lon},{origin_lat};{dest_lon},{dest_lat}"
        f"?overview=full&geometries=geojson"
    )
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    if data.get("code") != "Ok":
        return None
    return data["routes"][0]["geometry"]["coordinates"]

def snap(coord, decimals=5):
    return (round(coord[0], decimals), round(coord[1], decimals))

In [ ]:
segment_counter = Counter()

for v in tqdm(sample, desc="Routing"):
    lat, lon = v["lat"], v["lon"]

    dist2 = (hosp_lats - lat) ** 2 + (hosp_lons - lon) ** 2
    idx_k = np.argpartition(dist2, K_NEAREST)[:K_NEAREST]
    candidates = hospitals_df.iloc[idx_k].reset_index(drop=True)

    durations = osrm_table(lat, lon, candidates)

    best_idx, best_dur = None, float("inf")
    for i, dur in enumerate(durations):
        if dur is not None and dur < best_dur:
            best_dur, best_idx = dur, i

    best = candidates.iloc[best_idx]

    route_coords = osrm_route(lat, lon, best["lat"], best["lon"])

    for a, b in zip(route_coords[:-1], route_coords[1:]):
        segment_counter[(snap(a), snap(b))] += 1

    time.sleep(SLEEP_SEC)

print(f"Segmenti unici: {len(segment_counter):,}")

In [ ]:
rows = [
    {"count": cnt, "geometry": LineString([(a[0], a[1]), (b[0], b[1])])}
    for (a, b), cnt in segment_counter.items()
]
gdf = gpd.GeoDataFrame(rows, crs="EPSG:4326")

gdf["pct"] = gdf["count"].rank(pct=True) * 100

In [ ]:
lon_min, lat_min, lon_max, lat_max = regione.total_bounds

center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=8,
    tiles="CartoDB positron",
)

cmap = plt.get_cmap("RdYlBu_r")

pct_min = PERCENTILE_LOW
pct_max = 100
norm = mcolors.Normalize(vmin=pct_min, vmax=pct_max)

def pct_to_hex(pct):
    rgba = cmap(norm(np.clip(pct, pct_min, pct_max)))
    return mcolors.to_hex(rgba)

folium.GeoJson(
    regione,
    style_function= {
        "fillOpacity": 0,
        "color": "black",
        "weight": 1,
    },
).add_to(m)

gdf_sorted = gdf.sort_values("pct")

for _, row in gdf_sorted.iterrows():
    coords = [[lat, lon] for lon, lat in row.geometry.coords]
    pct = row["pct"]

    if pct < PERCENTILE_LOW:
        folium.PolyLine(
            coords,
            weight=0.8,
            color="grey",
            opacity=0.85,
        ).add_to(m)
    else:
        folium.PolyLine(
            coords,
            weight=2.5,
            color=pct_to_hex(pct),
            opacity=0.85,
        ).add_to(m)

for _, h in hospitals_df.iterrows():
    folium.CircleMarker(
        location=[h["lat"], h["lon"]],
        radius=7,
        color="white",
        weight=1.5,
        fill=True,
        fill_color="#2ecc71",
        fill_opacity=1.0,
    ).add_to(m)

m.save(OUTPUT_MAP)